In [1]:
# === Cell 1: Load and Combine Raw CSV Files ===

import pandas as pd
import os

print("--- Starting Data Pre-Processing Notebook ---")

# --- Define File Names ---
# These are the input files that should be in the same folder as this notebook.
coursera_input_filename = "coursera_reviews_log.csv"
udemy_input_filename = "udemy_reviews_log.csv"
edx_input_filename = "edx_reviews_log.csv"

# Initialize empty DataFrames to hold the data from each platform.
df_coursera = pd.DataFrame()
df_udemy = pd.DataFrame()
df_edx = pd.DataFrame()

# --- Load the datasets ---
print(f"\n--- Loading Raw Data ---")

# Load Coursera data
print(f"Attempting to load Coursera data from: {coursera_input_filename}")
if os.path.exists(coursera_input_filename):
    try:
        df_coursera = pd.read_csv(coursera_input_filename, encoding='utf-8-sig')
        df_coursera['Platform'] = 'Coursera'
        print(f"Successfully loaded {coursera_input_filename}: {df_coursera.shape[0]} rows")
    except Exception as e:
        print(f"[ERROR] Failed to load Coursera data: {e}")
else:
    print(f"[ERROR] File not found: {coursera_input_filename}")

# Load Udemy data
print(f"\nAttempting to load Udemy data from: {udemy_input_filename}")
if os.path.exists(udemy_input_filename):
    try:
        df_udemy = pd.read_csv(udemy_input_filename, encoding='utf-8-sig')
        df_udemy['Platform'] = 'Udemy'
        print(f"Successfully loaded {udemy_input_filename}: {df_udemy.shape[0]} rows")
    except Exception as e:
        print(f"[ERROR] Failed to load Udemy data: {e}")
else:
    print(f"[ERROR] File not found: {udemy_input_filename}")

# Load edX data
print(f"\nAttempting to load edX data from: {edx_input_filename}")
if os.path.exists(edx_input_filename):
    try:
        df_edx = pd.read_csv(edx_input_filename, encoding='utf-8-sig')
        df_edx['Platform'] = 'edX'
        print(f"Successfully loaded {edx_input_filename}: {df_edx.shape[0]} rows")
    except Exception as e:
        print(f"[ERROR] Failed to load edX data: {e}")
else:
    print(f"[ERROR] File not found: {edx_input_filename}")

# --- Combine DataFrames ---
# Concatenate the three individual DataFrames into one master DataFrame.
df_combined = pd.concat([df_coursera, df_udemy, df_edx], ignore_index=True)

# --- Verification ---
# Check if the combined DataFrame is not empty before proceeding.
if df_combined.empty:
    print("\n[ERROR] Combined DataFrame is empty. Halting further processing.")
else:
    print(f"\n--- Combined Data Summary ---")
    print(f"Total rows in combined DataFrame: {df_combined.shape[0]}")
    print("\nValue counts for 'Platform':")
    print(df_combined['Platform'].value_counts())
    print("\nFirst 5 rows of combined data:")
    display(df_combined.head())

--- Starting Data Pre-Processing Notebook ---

--- Loading Raw Data ---
Attempting to load Coursera data from: coursera_reviews_log.csv
Successfully loaded coursera_reviews_log.csv: 401 rows

Attempting to load Udemy data from: udemy_reviews_log.csv
Successfully loaded udemy_reviews_log.csv: 386 rows

Attempting to load edX data from: edx_reviews_log.csv
Successfully loaded edx_reviews_log.csv: 213 rows

--- Combined Data Summary ---
Total rows in combined DataFrame: 1000

Value counts for 'Platform':
Platform
Coursera    401
Udemy       386
edX         213
Name: count, dtype: int64

First 5 rows of combined data:


,Question_Log_URL,Full_Answer_Text,Date,Platform
0,https://www.quora.com/Is-a-Coursera-certificat...,It ultimately depends on your career path and ...,"March 25, 2025 at 3:56:42 PM",Coursera
1,https://www.quora.com/Is-a-Coursera-certificat...,"Yes, a Coursera certificate can definitely be ...","February 2, 2025 at 5:06:01 PM",Coursera
2,https://www.quora.com/Is-a-Coursera-certificat...,"Yes, a Coursera certificate can be quite usefu...","October 25, 2024 at 10:33:45 PM",Coursera
3,https://www.quora.com/Is-a-Coursera-certificat...,"According to today needs, Directcertify offers...","October 2, 2024 at 2:00:03 PM",Coursera
4,https://www.quora.com/Is-a-Coursera-certificat...,That certificate is nothing but rubbish.\nList...,"August 3, 2024 at 2:38:56 AM",Coursera


In [2]:
# === Cell 2: Filter DataFrame by Date to Match Project Scope ===

import pandas as pd

print("\n--- Filtering DataFrame by Date Scope (before 2025) ---")

# Define the cutoff date. We will keep all data strictly BEFORE this date.
cutoff_date = pd.to_datetime('2025-01-01')

# Ensure the combined DataFrame from Cell 1 exists and has the 'Date' column
if 'df_combined' in locals() and 'Date' in df_combined.columns:
    
    rows_before_filter = len(df_combined)
    print(f"Number of rows before date filtering: {rows_before_filter}")

    # --- Convert 'Date' column to datetime objects ---
    # This specifically handles the format "Month Day, Year at HH:MM:SS PM"
    # errors='coerce' will safely turn any dates that don't match the format into NaT (Not a Time)
    df_combined['Parsed_Date'] = pd.to_datetime(
        df_combined['Date'].str.replace(' at ', ' ', regex=False), 
        errors='coerce'
    )
    
    # Drop any rows where the date could not be parsed, just in case
    df_combined.dropna(subset=['Parsed_Date'], inplace=True)

    # --- Filter the DataFrame ---
    # Keep only the rows where the Parsed_Date is less than our cutoff date.
    df_in_scope = df_combined[df_combined['Parsed_Date'] < cutoff_date].copy()
    
    # --- Verification ---
    rows_after_filter = len(df_in_scope)
    removed_count = rows_before_filter - rows_after_filter
    
    print(f"Number of rows after filtering (before {cutoff_date.date()}): {rows_after_filter}")
    print(f"Number of rows removed (from 2025 and later): {removed_count}")

    # We can now drop the temporary 'Parsed_Date' column as it's no longer needed
    df_in_scope = df_in_scope.drop(columns=['Parsed_Date'])
    
    # --- Save intermediate result to CSV ---
    output_filename_1 = "preprocess_1.csv"
    df_in_scope.to_csv(output_filename_1, index=False, encoding='utf-8-sig')
    print(f"\n[SUCCESS] In-scope data saved to: {output_filename_1}")
    
else:
    print("[ERROR] Cannot perform date filtering. 'df_combined' DataFrame or 'Date' column not found.")


--- Filtering DataFrame by Date Scope (before 2025) ---
Number of rows before date filtering: 1000
Number of rows after filtering (before 2025-01-01): 687
Number of rows removed (from 2025 and later): 313

[SUCCESS] In-scope data saved to: preprocess_1.csv


C:\Users\afiqdisa\AppData\Local\Temp\ipykernel_3784\497782745.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_combined['Parsed_Date'] = pd.to_datetime(


In [3]:
# === Cell 3: Split Full Reviews into Individual Sentences ===

import nltk
import pandas as pd

# --- Download NLTK 'punkt' resource if necessary ---
# The 'punkt' tokenizer is NLTK's pre-trained model for splitting text into sentences.
try:
    nltk.data.find('tokenizers/punkt')
    print("[NLTK] 'punkt' resource already available.")
except LookupError:
    print("[NLTK] 'punkt' resource not found. Downloading...")
    nltk.download('punkt')
    print("[NLTK] Download complete.")

# --- Define Sentence Splitting Function ---
def split_into_sentences(text):
    """
    Safely splits a block of text into a list of sentences using NLTK.
    Handles non-string inputs to prevent errors.
    """
    if not isinstance(text, str):
        return [] # Return an empty list for non-string input
    return nltk.sent_tokenize(text)

# --- Load the data from the previous step ---
input_filename_2 = "preprocess_1.csv"
print(f"\n--- Loading in-scope data from: {input_filename_2} ---")
df_in_scope = pd.read_csv(input_filename_2)

# --- Apply Sentence Splitting ---
# We apply the function to the 'Full_Answer_Text' column.
print("\nSplitting full reviews into individual sentences...")
df_in_scope['Sentences'] = df_in_scope['Full_Answer_Text'].fillna('').astype(str).apply(split_into_sentences)

# --- Explode DataFrame to have one row per sentence ---
# The .explode() method transforms the DataFrame so that each sentence gets its own row.
df_sentences = df_in_scope.explode('Sentences', ignore_index=True)

# --- Clean up and Rename ---
# Remove any rows that might be empty after the explode operation.
df_sentences = df_sentences.dropna(subset=['Sentences'])
df_sentences = df_sentences[df_sentences['Sentences'].str.strip() != '']

# Rename the new column from 'Sentences' to 'Sentence_Text' for clarity.
df_sentences = df_sentences.rename(columns={'Sentences': 'Sentence_Text'})

# Reset the index for the new sentence-level DataFrame.
df_sentences = df_sentences.reset_index(drop=True)

# --- Save intermediate result to CSV ---
# This saves the output of this step, allowing for easy inspection.
output_filename_2 = "preprocess_2.csv"
df_sentences.to_csv(output_filename_2, index=False, encoding='utf-8-sig')
print(f"\n[SUCCESS] Intermediate data saved to: {output_filename_2}")

# --- Verification ---
print(f"Shape after sentence splitting: {df_sentences.shape}")
print("\nFirst 5 rows of the new sentence-level data:")
display(df_sentences[['Platform', 'Sentence_Text']].head())

[NLTK] 'punkt' resource already available.

--- Loading in-scope data from: preprocess_1.csv ---

Splitting full reviews into individual sentences...

[SUCCESS] Intermediate data saved to: preprocess_2.csv
Shape after sentence splitting: (7912, 5)

First 5 rows of the new sentence-level data:


,Platform,Sentence_Text
0,Coursera,"Yes, a Coursera certificate can be quite useful."
1,Coursera,While it may not hold the same weight as a deg...
2,Coursera,"Many employers recognize Coursera courses, esp..."
3,Coursera,"Additionally, completing these courses can hel..."
4,Coursera,If you're looking for more certification optio...


In [4]:
# === Cell 4: Basic Text Cleaning ===

import pandas as pd
import re
import string

# --- Define Cleaning Function ---
def clean_text(text):
    """
    Converts text to lowercase, removes punctuation, removes numbers,
    and strips extra whitespace.
    """
    # Convert to lowercase to ensure consistency (e.g., "Course" and "course" are treated as the same).
    text = str(text).lower()
    
    # Remove all punctuation characters.
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove any standalone numbers or numbers within words.
    text = re.sub(r'\d+', '', text)
    
    # Remove any leading/trailing whitespace and collapse multiple spaces into one.
    text = ' '.join(text.split())
    
    return text

# --- Load the data from the previous step ---
input_filename_3 = "preprocess_2.csv"
print(f"\n--- Loading sentence-level data from: {input_filename_3} ---")
df_sentences = pd.read_csv(input_filename_3)

# --- Apply Cleaning ---
# Apply the function to the 'Sentence_Text' column to create a new 'Cleaned_Text' column.
print("\nCleaning sentence text (lowercase, remove punctuation/numbers)...")
df_sentences['Cleaned_Text'] = df_sentences['Sentence_Text'].apply(clean_text)

# --- Save intermediate result to CSV ---
# The output of this cleaning step is saved with the new naming convention.
output_filename_3 = "preprocess_3.csv"
df_sentences.to_csv(output_filename_3, index=False, encoding='utf-8-sig')
print(f"\n[SUCCESS] Intermediate data saved to: {output_filename_3}")

# --- Verification ---
print("\nVerification: Showing original vs. cleaned text:")
display(df_sentences[['Sentence_Text', 'Cleaned_Text']].head())


--- Loading sentence-level data from: preprocess_2.csv ---

Cleaning sentence text (lowercase, remove punctuation/numbers)...

[SUCCESS] Intermediate data saved to: preprocess_3.csv

Verification: Showing original vs. cleaned text:


,Sentence_Text,Cleaned_Text
0,"Yes, a Coursera certificate can be quite useful.",yes a coursera certificate can be quite useful
1,While it may not hold the same weight as a deg...,while it may not hold the same weight as a deg...
2,"Many employers recognize Coursera courses, esp...",many employers recognize coursera courses espe...
3,"Additionally, completing these courses can hel...",additionally completing these courses can help...
4,If you're looking for more certification optio...,if youre looking for more certification option...


In [5]:
# === Cell 5: Stop Word Removal ===

import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# --- Download NLTK 'stopwords' resource if necessary ---
try:
    nltk.data.find('corpora/stopwords')
    print("[NLTK] 'stopwords' resource already available.")
except LookupError:
    print("[NLTK] 'stopwords' resource not found. Downloading...")
    nltk.download('stopwords')
    print("[NLTK] Download complete.")

# --- Define Stop Word Removal Function ---
# Load the standard set of English stop words once to be efficient.
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    """
    Removes standard English stop words from a string of text.
    """
    # Tokenize the text into individual words.
    words = word_tokenize(str(text))
    
    # Create a new list of words, keeping only those that are NOT in the stop_words set.
    filtered_words = [word for word in words if word not in stop_words]
    
    # Join the filtered words back into a single string.
    return ' '.join(filtered_words)

# --- Load the data from the previous step ---
input_filename_4 = "preprocess_3.csv"
print(f"\n--- Loading cleaned data from: {input_filename_4} ---")
df_sentences = pd.read_csv(input_filename_4)

# --- Apply Stop Word Removal ---
# Apply the function to the 'Cleaned_Text' column.
print("\nRemoving stop words...")
df_sentences['Text_No_Stopwords'] = df_sentences['Cleaned_Text'].apply(remove_stopwords)

# --- Save intermediate result to CSV ---
output_filename_4 = "preprocess_4.csv"
df_sentences.to_csv(output_filename_4, index=False, encoding='utf-8-sig')
print(f"\n[SUCCESS] Intermediate data saved to: {output_filename_4}")

# --- Verification ---
print("\nVerification: Showing cleaned text vs. text without stop words:")
display(df_sentences[['Cleaned_Text', 'Text_No_Stopwords']].head())

[NLTK] 'stopwords' resource already available.

--- Loading cleaned data from: preprocess_3.csv ---

Removing stop words...

[SUCCESS] Intermediate data saved to: preprocess_4.csv

Verification: Showing cleaned text vs. text without stop words:


,Cleaned_Text,Text_No_Stopwords
0,yes a coursera certificate can be quite useful,yes coursera certificate quite useful
1,while it may not hold the same weight as a deg...,may hold weight degree demonstrates commitment...
2,many employers recognize coursera courses espe...,many employers recognize coursera courses espe...
3,additionally completing these courses can help...,additionally completing courses help gain prac...
4,if youre looking for more certification option...,youre looking certification options could comp...


In [6]:
# === Cell 6: Lemmatization with Part-of-Speech (POS) Tagging ===

import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# --- Download NLTK resources if necessary ---
try:
    nltk.data.find('corpora/wordnet')
    nltk.data.find('taggers/averaged_perceptron_tagger')
    print("[NLTK] Required resources ('wordnet', 'averaged_perceptron_tagger') already available.")
except LookupError:
    print("[NLTK] One or more resources not found. Downloading...")
    nltk.download('wordnet')
    nltk.download('averaged_perceptron_tagger')
    print("[NLTK] Download complete.")

# --- Helper function to map NLTK POS tags to WordNet POS tags ---
def get_wordnet_pos(treebank_tag):
    """
    Maps NLTK's Part-of-Speech tags to a format that the WordNetLemmatizer
    understands, improving the accuracy of the lemmatization.
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        # Default to noun if the tag is not recognized.
        return wordnet.NOUN

# --- Define Lemmatization Function ---
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    """
    Performs lemmatization on a string of text using POS tagging for accuracy.
    """
    words = word_tokenize(str(text))
    pos_tags = pos_tag(words)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_tags]
    return ' '.join(lemmatized_words)

# --- Load the data from the previous step ---
input_filename_5 = "preprocess_4.csv"
print(f"\n--- Loading data from: {input_filename_5} ---")
df_sentences = pd.read_csv(input_filename_5)

# --- Apply Lemmatization ---
print("\nLemmatizing text (using POS tags)...")
df_sentences['Lemmatized_Text'] = df_sentences['Text_No_Stopwords'].apply(lemmatize_text)

# --- Save the FINAL preprocessed data ---
output_filename_final = "preprocess_final.csv"
# We only save the columns needed for the next notebook to keep the file clean.
columns_to_save = ['Platform', 'Question_Log_URL', 'Date', 'Sentence_Text', 'Lemmatized_Text']
df_sentences[columns_to_save].to_csv(output_filename_final, index=False, encoding='utf-8-sig')
print(f"\n[SUCCESS] Final preprocessed data saved to: {output_filename_final}")

# --- Verification ---
print("\nVerification: Showing text before vs. after lemmatization:")
display(df_sentences[['Text_No_Stopwords', 'Lemmatized_Text']].head())

[NLTK] One or more resources not found. Downloading...


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\afiqdisa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\afiqdisa\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


[NLTK] Download complete.

--- Loading data from: preprocess_4.csv ---

Lemmatizing text (using POS tags)...

[SUCCESS] Final preprocessed data saved to: preprocess_final.csv

Verification: Showing text before vs. after lemmatization:


,Text_No_Stopwords,Lemmatized_Text
0,yes coursera certificate quite useful,yes coursera certificate quite useful
1,may hold weight degree demonstrates commitment...,may hold weight degree demonstrate commitment ...
2,many employers recognize coursera courses espe...,many employer recognize coursera course especi...
3,additionally completing courses help gain prac...,additionally complete course help gain practic...
4,youre looking certification options could comp...,youre look certification option could compleme...
